# Single-Project Pipeline Runner

Run the PAD2Skills pipeline **end-to-end for each project**: for each selected project, run *all stages* (PDF → Markdown, sectioning, summarization, extraction, matching, etc.) before moving on to the next project.

This is intentionally different from the **multi-project pipeline** (notebook 98), which runs each stage for *all projects* before moving to the next stage.

## 0. Setup

### 0.01 Import Required Libraries

In [16]:
import os
from pathlib import Path
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import time

import pandas as pd

# Import our config and pipeline
import sys
sys.path.append(str(Path.cwd().parent))
from src.config import load_config
from src.pipeline.single_project_pipeline import SingleProjectPipeline

### 0.02 Load Configuration and Environment Variables

In [17]:
# Load environment variables from .env file
project_root = Path.cwd().parent
env_path = project_root / ".env"

if not env_path.exists():
    raise FileNotFoundError(
        f"'.env' file not found at {env_path}\n"
        "Please copy .env.example to .env and add your OpenAI API key."
    )

# Load from specific path
load_dotenv(env_path, override=True)

# Load project config
config = load_config()

# Get OpenAI API key from environment
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Verify API key is set
if not OPENAI_API_KEY:
    raise ValueError("Missing required environment variable: OPENAI_API_KEY")

print("✓ Environment variables loaded")
print(f"  API Key: {OPENAI_API_KEY[:10]}...{OPENAI_API_KEY[-4:]}")

✓ Environment variables loaded
  API Key: sk-proj-cj...__0A


### 0.03 Set Up Paths

In [18]:
# Get paths
project_data_dir = project_root / "data" / "bronze" / "project_data"
all_project_details_path = project_data_dir / "all_project_details.csv"
project_summary_path = project_data_dir / "project_summary.csv"

print(f"Project data directory: {project_data_dir}")
print(f"All project details path: {all_project_details_path}")
print(f"Project summary path: {project_summary_path}")
print(f"Files exist: {all_project_details_path.exists()} / {project_summary_path.exists()}")

Project data directory: /Users/lauren/repos/PAD2Skills/data/bronze/project_data
All project details path: /Users/lauren/repos/PAD2Skills/data/bronze/project_data/all_project_details.csv
Project summary path: /Users/lauren/repos/PAD2Skills/data/bronze/project_data/project_summary.csv
Files exist: True / True


### 0.04 Set Overwrite Flags

Configure which pipeline steps should overwrite existing files. Default is `False` for all (skip existing data).

In [19]:
# Overwrite flags for each pipeline step
# Set to True to force regeneration of files
ow_pdf = False
ow_sections = False
ow_abbr = False
ow_chunks = False
ow_long_summary = False
ow_short_summary = False
ow_occupations = False
ow_occs_csv = False
ow_esco_prep = False
ow_esco_match = False
ow_esco_select = False
ow_unique_esco = False
ow_nace_prep = False
ow_nace_select = False
ow_skills = False
ow_onet_prep = False
ow_onet_merge = False

print("Overwrite flags configured")

Overwrite flags configured


## 1. Get Project Details

### 1.01 Load Project Data

In [20]:
# Read project details and summary
all_project_details = pd.read_csv(all_project_details_path)
project_summary = pd.read_csv(project_summary_path)

print(f"All project details: {all_project_details.shape[0]} rows, {all_project_details.shape[1]} columns")
print(f"Project summary: {project_summary.shape[0]} rows, {project_summary.shape[1]} columns")

All project details: 123 rows, 20 columns
Project summary: 123 rows, 4 columns


### 1.02 Convert column names to snake_case

In [21]:
# Convert column names to snake_case
all_project_details.columns = (
    all_project_details.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^\w]', '_', regex=True)
    .str.replace(r'_+', '_', regex=True)
    .str.strip('_')
)

print("Column names after standardization:")
print(list(all_project_details.columns))

Column names after standardization:
['project_id', 'status', 'team_leader', 'borrower_2', 'country', 'disclosure_date', 'approval_date', 'effective_date', 'total_project_cost_1', 'implementing_agency', 'region', 'fiscal_year_3', 'commitment_amount', 'environmental_category', 'environmental_and_social_risk', 'closing_date', 'last_stage_reached', 'last_update_date', 'consultant_services_required', 'associated_projects']


### 1.03 Merge and Filter Projects

In [22]:
# Merge project_summary to all_project_details
all_projects = all_project_details.merge(
    project_summary,
    on="project_id",
    how="left"
)

print(f"Merged data: {all_projects.shape[0]} rows, {all_projects.shape[1]} columns")

# Keep only projects with downloaded PADs
all_projects = all_projects[all_projects["pads_downloaded"] >= 1]

print(f"Projects with downloaded PADs: {all_projects.shape[0]} rows")
print(f"\nFirst few projects:")
print(all_projects.head())

Merged data: 123 rows, 23 columns
Projects with downloaded PADs: 98 rows

First few projects:
  project_id  status                                        team_leader  \
1    P119893  Closed                    Abdulhakim Mohammed Abdisubhan    
3    P173506  Active  Didier Makoso Tsasa , Fabrice Karl Bertholet, ...   
4    P176731  Active  Janina Franco , Abdulhakim Mohammed Abdisubhan...   
5    P507759  Active                     Jenny Jing Chao , Maria Arango   
6    P180547  Active   Monali Ranade , Dana Rysankova, Alona Kazantseva   

                                          borrower_2  \
1            Federal Democratic Republic of Ethiopia   
3                       DEMOCRATIC REPUBLIC OF CONGO   
4            Federal Democratic Republic of Ethiopia   
5                             Republic of Mozambique   
6  Common Market for Eastern and Southern Africa ...   

                         country    disclosure_date  \
1                       Ethiopia  December 22, 2011   
3  Congo

## 2. Select Projects

### 2.01 Select Projects to Process

Choose which projects to run through the pipeline. You can select the first N projects or define a custom list.

In [23]:
# Option 1: Select first N projects
# selected_projects = all_projects["project_id"][0:10].tolist()

# Option 2: Custom list (uncomment to use)
# selected_projects = ['P119893', 'P173506', 'P176731', 'P507759', 'P180547']
selected_projects = ['P164435']

# Option 3: All projects
#selected_projects = all_projects["project_id"].tolist()

print(f"Selected {len(selected_projects)} projects:")
for project_id in selected_projects:
    print(f"  {project_id}")

Selected 1 projects:
  P164435


### 2.02 Filter Projects DataFrame

In [24]:
# Filter projects dataframe for selected projects
projects_df = all_projects[all_projects["project_id"].isin(selected_projects)]

print(f"Filtered to {len(projects_df)} projects:")
print(projects_df[["project_id", "status", "country"]].to_string(index=False))

Filtered to 1 projects:
project_id status country
   P164435 Active Burundi


## 3. Run Pipeline for Each Project

### 3.01 Process All Selected Projects

Run the complete pipeline end-to-end for each project. Each project will go through all 13 steps before moving to the next project.

In [25]:
def process_project(project_id, worker_id, ow_flags):
    """
    Process a single project through the complete pipeline.
    
    Args:
        project_id: The project ID to process
        worker_id: Worker index (0-3) for staggered starts
        ow_flags: Dictionary of overwrite flags
        
    Returns:
        Tuple of (project_id, success, error_message, timing_data)
    """
    # Stagger worker starts by 1 second to reduce simultaneous API requests
    time.sleep(worker_id)
    
    try:
        # Create pipeline instance
        pipeline = SingleProjectPipeline(
            project_id=project_id,
            print_progress=True,
            **ow_flags
        )
        
        # Run the pipeline
        pipeline.run()
        
        # Check if any steps failed
        df = pd.DataFrame(pipeline.timing_data)
        failed_steps = df["error_occurred"].sum()
        
        if failed_steps > 0:
            return (project_id, False, f"{failed_steps} steps failed", pipeline.timing_data)
        else:
            return (project_id, True, None, pipeline.timing_data)
            
    except Exception as e:
        return (project_id, False, str(e), [])

In [26]:
# Collect overwrite flags for worker function
ow_flags = {
    "ow_pdf": ow_pdf,
    "ow_sections": ow_sections,
    "ow_abbr": ow_abbr,
    "ow_chunks": ow_chunks,
    "ow_long_summary": ow_long_summary,
    "ow_short_summary": ow_short_summary,
    "ow_occupations": ow_occupations,
    "ow_occs_csv": ow_occs_csv,
    "ow_esco_prep": ow_esco_prep,
    "ow_esco_match": ow_esco_match,
    "ow_esco_select": ow_esco_select,
    "ow_unique_esco": ow_unique_esco,
    "ow_nace_prep": ow_nace_prep,
    "ow_nace_select": ow_nace_select,
    "ow_skills": ow_skills,
    "ow_onet_prep": ow_onet_prep,
    "ow_onet_merge": ow_onet_merge,
}

# Track which projects completed successfully
completed_projects = []
failed_projects = []
all_timing_data = []

print(f"\n{'=' * 80}")
print(f"PROCESSING {len(selected_projects)} PROJECTS WITH 4 WORKERS")
print(f"{'=' * 80}\n")

# Run pipeline in parallel with 4 workers
with ThreadPoolExecutor(max_workers=4) as executor:
    # Submit all projects to the executor
    futures = {
        executor.submit(process_project, project_id, idx % 4, ow_flags): project_id
        for idx, project_id in enumerate(selected_projects)
    }
    
    # Process completed projects with progress bar
    with tqdm(total=len(selected_projects), desc="Projects") as pbar:
        for future in as_completed(futures):
            project_id, success, error_message, timing_data = future.result()
            
            if success:
                completed_projects.append(project_id)
                print(f"✓ {project_id} completed successfully")
            else:
                failed_projects.append((project_id, error_message))
                print(f"✗ {project_id} failed: {error_message}")
            
            # Collect timing data
            all_timing_data.extend(timing_data)
            
            # Update progress bar
            pbar.update(1)

print(f"\n{'=' * 80}")
print("ALL PROJECTS PROCESSED")
print(f"{'=' * 80}")
print(f"Completed: {len(completed_projects)}/{len(selected_projects)}")
print(f"Failed: {len(failed_projects)}/{len(selected_projects)}")


PROCESSING 1 PROJECTS WITH 4 WORKERS


SINGLE-PROJECT PIPELINE: P164435
P164435 Step 01_pdf (PDF Conversion): 0.0 seconds


Projects:   0%|          | 0/1 [00:00<?, ?it/s]

P164435 Step 02_sections (Extract Sections): 0.0 seconds
P164435 Step 03_abbr (Extract Abbreviations): 0.0 seconds
P164435 Step 04_chunks (Create Chunks): 0.0 seconds
P164435 Step 05_long_summary (Generate Long Summary): 0.0 seconds
P164435 Step 05b_short_summary (Generate Short Summary): 0.0 seconds
P164435 Step 06_occupations (Extract Occupations): 0.0 seconds
P164435 Step 06b_occs_csv (Prepare Occupations CSV): 0.0 seconds
P164435 Step 07_esco_prep (Prepare ESCO Data): 0.0 seconds
Output already exists: /Users/lauren/repos/PAD2Skills/data/silver/esco_matching_csv/P164435_esco_matches.csv
Use overwrite=True to force re-matching
P164435 Step 07b_esco_match (Match to ESCO): 0.0 seconds
Found 2 JSON chunk files for project P164435
[1/2] Skipping existing: P164435_000-074_esco_selection.json
[2/2] Processing: P164435_075-092_esco_matches.json
  Chunk ID: 075-092
  Records: 18


2026-01-15 07:50:51,384 - INFO - Use pytorch device_name: cpu
2026-01-15 07:50:51,384 - INFO - Load pretrained SentenceTransformer: intfloat/e5-small-v2


  ✓ Saved to: P164435_075-092_esco_selection.json
✓ Processed 2 chunks
✓ Results saved to: /Users/lauren/repos/PAD2Skills/data/silver/choose_esco_json
✓ Loaded original PAD data: 93 rows
Loading 2 JSON selection files...
✓ Combined 93 records from 2 files
✓ Saved combined selections to: /Users/lauren/repos/PAD2Skills/data/silver/choose_esco_csv/P164435_esco_selections.csv
  Rows: 93, Columns: 12
P164435 Step 08_esco_select (Select Best ESCO): 44.5 seconds
✓ Loaded selections data: 93 rows
  Columns: ['project_id', 'record_id', 'esco_id', 'esco_label', 'rank', 'confidence', 'needs_manual_review', 'pad_occupation', 'pad_activity', 'pad_skills', 'pad_quote', 'pad_section_id']
✓ Mapped section names: 93 records

✓ Filtered out records with needs_manual_review=True
  Dropped: 0 rows
  Remaining: 93 rows

✓ Formatted pad_quote with section names

✓ Flattened data by esco_id
  Unique ESCO IDs: 45
✓ Created esco_uri field

✓ Loaded ESCO occupations data: 3043 rows

✓ Merged with ESCO data
  Fi

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

✓ Encoded 284 NACE groups (shape: (284, 384))
✓ Created group_code to index mapping

Preparing 45 unique ESCO texts for embedding
Encoding ESCO occupations...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Encoded 45 ESCO occupations (shape: (45, 384))
✓ Created esco_id to index mapping

Selecting best NACE group for each ESCO ID...
✓ Selected best NACE group for 45 ESCO IDs
  ESCO IDs with a best group: 45 (100.0%)
  ESCO IDs with no group: 0
  Mean similarity score: 0.8060
  Median similarity score: 0.8088

Merging results with ESCO data...
✓ Merged results
  Rows with NACE group: 45 (100.0%)

✓ Saved results to: /Users/lauren/repos/PAD2Skills/data/silver/unique_esco_nace_csv/P164435_unique_matched_with_nace.csv
  Total rows: 45
  Unique ESCO IDs: 45
  ESCO IDs with NACE group: 45
  Unique NACE sections: 14
  Unique NACE divisions: 19
  Unique NACE groups: 24
P164435 Step 09b_nace_select (Select NACE Groups): 22.0 seconds
ESCO Skills Refiner
Project ID: P164435
Chunk size: 3 occupations

Loading data files...
  ✓ Loaded 45 unique ESCO occupations
  ✓ Loaded 126051 occupation-skill relations
  ✓ Filtered to 67600 essential skills
  ✓ Loaded project summary (1735 characters)

Merging o

## 4. Analyze Timing Results

### 4.01 Load All Timing CSVs

In [27]:
# Load all timing CSV files
timing_dir = project_root / "data" / "silver" / "zz_status_timing"

# Find timing files for selected projects
timing_files = []
for project_id in selected_projects:
    project_files = list(timing_dir.glob(f"{project_id}_*_timing.csv"))
    if project_files:
        # Get the most recent file for this project
        latest_file = max(project_files, key=lambda p: p.stat().st_mtime)
        timing_files.append(latest_file)

print(f"Found {len(timing_files)} timing files")

# Load and combine all timing data
all_timing_data = []
for timing_file in timing_files:
    df = pd.read_csv(timing_file)
    project_id = timing_file.name.split("_")[0]
    df["project_id"] = project_id
    all_timing_data.append(df)

if all_timing_data:
    combined_timing = pd.concat(all_timing_data, ignore_index=True)
    print(f"\nCombined timing data: {len(combined_timing)} rows")
else:
    print("\n⚠ No timing data found")
    combined_timing = None

Found 1 timing files

Combined timing data: 17 rows


### 4.02 Summary Statistics by Project

In [28]:
if combined_timing is not None:
    # Calculate summary statistics by project
    project_summary = combined_timing.groupby("project_id").agg({
        "elapsed_minutes": "sum",
        "error_occurred": "sum",
        "step_code": "count"
    }).rename(columns={
        "elapsed_minutes": "total_time_minutes",
        "error_occurred": "failed_steps",
        "step_code": "total_steps"
    })
    
    project_summary["success_rate"] = (
        (project_summary["total_steps"] - project_summary["failed_steps"]) 
        / project_summary["total_steps"] * 100
    )
    
    print("\nSummary by Project:")
    print("=" * 80)
    print(project_summary.to_string())
    
    print(f"\n\nOverall Statistics:")
    print("=" * 80)
    print(f"Total projects: {len(project_summary)}")
    print(f"Total time: {project_summary['total_time_minutes'].sum():.2f} minutes")
    print(f"Average time per project: {project_summary['total_time_minutes'].mean():.2f} minutes")
    print(f"Total failed steps: {int(project_summary['failed_steps'].sum())}")
    print(f"Average success rate: {project_summary['success_rate'].mean():.1f}%")


Summary by Project:
            total_time_minutes  failed_steps  total_steps  success_rate
project_id                                                             
P164435                  27.16             0           17         100.0


Overall Statistics:
Total projects: 1
Total time: 27.16 minutes
Average time per project: 27.16 minutes
Total failed steps: 0
Average success rate: 100.0%


### 4.03 Summary Statistics by Step

In [29]:
if combined_timing is not None:
    # Calculate summary statistics by step
    step_summary = combined_timing.groupby(["step_code", "step_name"]).agg({
        "elapsed_minutes": ["mean", "median", "min", "max"],
        "error_occurred": "sum",
        "project_id": "count"
    }).round(2)
    
    step_summary.columns = ["avg_minutes", "median_minutes", "min_minutes", "max_minutes", "failures", "total_runs"]
    step_summary["failure_rate"] = (step_summary["failures"] / step_summary["total_runs"] * 100).round(1)
    
    print("\nSummary by Step:")
    print("=" * 80)
    print(step_summary.to_string())


Summary by Step:
                                               avg_minutes  median_minutes  min_minutes  max_minutes  failures  total_runs  failure_rate
step_code         step_name                                                                                                             
01_pdf            PDF Conversion                      0.00            0.00         0.00         0.00         0           1           0.0
02_sections       Extract Sections                    0.00            0.00         0.00         0.00         0           1           0.0
03_abbr           Extract Abbreviations               0.00            0.00         0.00         0.00         0           1           0.0
04_chunks         Create Chunks                       0.00            0.00         0.00         0.00         0           1           0.0
05_long_summary   Generate Long Summary               0.00            0.00         0.00         0.00         0           1           0.0
05b_short_summary Gener

### 4.04 Failed Steps Details

In [30]:
if combined_timing is not None:
    # Show details of failed steps
    failed_steps = combined_timing[combined_timing["error_occurred"] == True]
    
    if len(failed_steps) > 0:
        print(f"\nFailed Steps Details ({len(failed_steps)} failures):")
        print("=" * 80)
        print(failed_steps[["project_id", "step_code", "step_name", "error_message"]].to_string(index=False))
    else:
        print("\n✓ No failed steps!")


✓ No failed steps!
